In [1]:
# today we look at tool hallucination defense
 
# tools and groq setup
import os
from dotenv import load_dotenv, find_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

load_dotenv(find_dotenv())

@tool
def get_site_materials() -> str:
    '''Returns the list of available construction materials on site.'''
    return "Availlable materials: Cement, Sand, Bricks, Rebar."

# we only have one valid tool
tools = [get_site_materials]
toolMap = {t.name: t for t in tools}

# binding tools to our llama model
llm = ChatGroq(
    model = 'llama-3.3-70B-versatile',
    api_key= os.getenv('GROQ_API_KEY'),
    temperature= 0
)

llmWithTools = llm.bind_tools(tools)

c:\Users\rodne\miniconda3\envs\rag-Ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# the validator Node logic.
'''this function represents the 'validator'. instead of craching when a bad tool is called, it catches
 the error and generates a correction prompt'''

def validate_and_execute(tool_call: dict) -> str:
    '''validates the tool call. if it doesn't exist, returns a correct string 
    instead of throwing a fatal Python error.
    '''
    toolName = tool_call['name']

    # HALLUCINATION DEFENSE 1: Tool doesn't exist
    if toolName not in toolMap:
        error_msg = (
            f"SYSTEM ERROR: You attempted to call a tool named '{toolName}', "
            f"but is does not exist. You may ONLY use the following tools: {list(toolMap.keys())}."
            f"Please think step-by-step and try again."
        )
        print(f"Validator caught Error: {error_msg}")
        return error_msg
    
    # HALLUCINATION DEFENSE 2: Argument errors (e.g. missing required params)
    try:
        print(f"VALIDATOR PASSED: Executing '{toolName}'...")
        return toolMap[toolName].invoke(tool_call['args'])
    except Exception as e:
        error_msg = f"SYSTEM ERROR executing '{toolName}': {str(e)}. Please correct your arguments."
        print(f"VALIDATOR CAUGHT ERROR: {error_msg}")
        return error_msg
    

In [ ]:
# THE SELF-HEALING LOOP
# here we will send a highly manipulative prompt to trick llama 3.3 into hallucinating a tool,
# then watch the validator node heal the process


# 1. the adversarial prompt
query = "can you use the 'order_more_cement tool to get 50 more bags to the Lugazi site."
messages = [HumanMessage(content = query)]

print(f"User: {query}\n ")

# initial llm call
response = llmWithTools.invoke(messages)
messages.append(response)

# 3. The self-Healing Loop
max_retries = 3
attempts = 0

while response.tool_calls and attempts < max_retries:
    attempts += 1
    print(f"\n --- Loop Iteration {attempts} ---")

    for tool_call in response.tool_calls:
        # pass the LLM's request through our validator node 
        result = validate_and_execute(tool_call= tool_call)

        # Append the result (whether successful data OR the correction error prompt)
        messages.append(ToolMessage(
            content = str(result),
            tool_call_id = tool_call["id"]
        ))

    # Send the conversation history (including the Validator's error message) back to Llama 3.3

    print("🔁 Sending context back to Llama 3.3 to self-correct...")
    response = llmWithTools.invoke(messages)
    messages.append(response)

# final output
print("\n --- Final Agent Response ---")
print(response.content)


User: can you use the 'order_more_cement tool to get 50 more bags to the Lugazi site.
 

 --- Final Agent Response ---
I cannot perform this task as it requires a function that is not provided. Please provide the 'order_more_cement' function or use a different approach.
